In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "1"

In [3]:
from llg_emulator.data import LLGStepperSource
from llg_emulator.model import LLGEmulator
import jax.random as jr
from llg_emulator.training import correlation_epoch
from pathlib import Path
import matplotlib.pyplot as plt
import jax.numpy as jnp

/data3/smith/work/llg-emulator/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-06-11 08:43:33 NeuralMag:INFO [NeuralMag] Version 0.9.4
INFO:NeuralMag:[NeuralMag] Version 0.9.4


In [4]:
path = Path("../micromagnetic-data/data/dynamics/small/val")
source = LLGStepperSource(path=path)

100%|██████████| 14/14 [00:02<00:00,  5.34it/s]


In [5]:
key = jr.PRNGKey(0)
model = LLGEmulator(key=key)

In [6]:
corr_mean, corr_std = correlation_epoch(model, source)

In [7]:
corr_mean.shape, corr_std.shape

((101,), (101,))

In [8]:
# plt.plot(range(corr_mean.shape[0]), corr_mean)
# plt.fill_between(
#     range(corr_mean.shape[0]),
#     corr_mean + corr_std,
#     corr_mean - corr_std,
#     alpha=0.3,
# )
# plt.ylim((0, 1))
# plt.show()

In [9]:
import plotly.graph_objects as go
import numpy as np

x = np.arange(corr_mean.shape[0])

fig = go.Figure()

# Mean line
fig.add_trace(
    go.Scatter(
        x=x,
        y=corr_mean,
        mode="lines",
        name="Mean",
    )
)

# Shaded ± std region
fig.add_trace(
    go.Scatter(
        x=np.concatenate([x, x[::-1]]),
        y=np.concatenate([corr_mean + corr_std, (corr_mean - corr_std)[::-1]]),
        fill="toself",
        fillcolor="rgba(0,100,80,0.3)",
        line=dict(color="rgba(255,255,255,0)"),
        hoverinfo="skip",
        showlegend=False,
    )
)

fig.update_yaxes(range=[0, 1])

fig.show()